# External emotion evaluation — test split only

Breaks the internal-encoder circularity flagged by reviewer: evaluates generated videos with **classifiers never involved in training or model selection**.

**Protocol:**
1. For each fine-tuned checkpoint (baseline + ablation configs), generate full-length videos from **test split** audio.
2. Run independent classifiers — all trained on non-RAVDESS data — on the generated frames.
3. Compare external accuracy / F1 against the ground-truth emotion label.
4. Report Δ external-F1 vs baseline with statistical significance.

**External models:**
- **Video**: `trpakov/vit-face-expression` — ViT-base trained on AffectNet (7 emotions, incl. happy/angry/disgust).
- **Audio (optional dim.)**: `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` — MSP-Podcast dimensional emotion (arousal/dominance/valence) for cross-modal sanity check.

Neither was used at any stage of training or model selection.

In [ ]:
!pip install -q transformers librosa scipy scikit-learn Pillow

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/Wav2Lip")

import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from PIL import Image
from scipy import stats
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    AutoFeatureExtractor, AutoModel,
)

from models.wav2lip import Wav2Lip as Wav2LipModel

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
WAV2LIP_CKPT = "/content/Wav2Lip/checkpoints/wav2lip.pth"

ABLATION_DIR = Path("/content/wav2lip_loss_ablation")
ORIG_DIR = Path("/content/wav2lip_finetuned")
OUT_DIR = Path("/content/external_eval_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE = {0, 1, 3, 5, 7}
REMAP = {2: 0, 4: 1, 6: 2}
EMOTIONS = ["happy", "angry", "disgust"]
NUM_EMO = len(EMOTIONS)

print(f"Device: {DEVICE}")

## Checkpoints to evaluate

Edit this list to match the runs you want to report. Each entry is `(display_name, checkpoint_path)`.

In [ ]:
CHECKPOINTS = [
    # Loss-ablation configs (from 04b_wav2lip_loss_ablation.ipynb)
    ("baseline",  ABLATION_DIR / "abl-baseline"  / "wav2lip.pth"),
    ("cos-only",  ABLATION_DIR / "abl-cos-only"  / "wav2lip.pth"),
    ("ce-only",   ABLATION_DIR / "abl-ce-only"   / "wav2lip.pth"),
    ("ce-cos",    ABLATION_DIR / "abl-ce-cos"    / "wav2lip.pth"),
    ("ce-kl",     ABLATION_DIR / "abl-ce-kl"     / "wav2lip.pth"),
]
CHECKPOINTS = [(n, p) for n, p in CHECKPOINTS if Path(p).is_file()]
print(f"Evaluating {len(CHECKPOINTS)} checkpoints:")
for n, p in CHECKPOINTS:
    print(f"  {n:20s} -> {p}")

## Load external classifier (video — AffectNet-trained ViT)

AffectNet emotion labels need mapping to our 3-class set:
- `happy` → `happy` (0)
- `angry` → `angry` (1)
- `disgust` → `disgust` (2)
- others (`neutral`, `sad`, `fear`, `surprise`) are tracked separately for confusion analysis.

In [ ]:
EXT_VIDEO_MODEL = "trpakov/vit-face-expression"
ext_video_proc = AutoImageProcessor.from_pretrained(EXT_VIDEO_MODEL)
ext_video_model = AutoModelForImageClassification.from_pretrained(EXT_VIDEO_MODEL).to(DEVICE).eval()
for p in ext_video_model.parameters():
    p.requires_grad = False

ext_id2label = ext_video_model.config.id2label
ext_label2id = {v.lower(): k for k, v in ext_id2label.items()}
print("External video classifier labels:", ext_id2label)

# Map our 3 target classes to AffectNet indices
TARGET_EXT_IDS = {
    "happy":   ext_label2id["happy"],
    "angry":   ext_label2id["angry"],
    "disgust": ext_label2id["disgust"],
}
print("Target→ext mapping:", TARGET_EXT_IDS)

## Test dataset — full-length videos

Generates all 32 preprocessed frames per test sample for richer external-classifier signal (vs. the random 5-frame windows used in training).

In [ ]:
IMG_SIZE = 96
MEL_STEP = 16
SR = 16000
FPS = 25


def wav_to_mel(wav_path, sr=SR):
    y, _ = librosa.load(wav_path, sr=sr)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=80, hop_length=200, win_length=800, fmin=55, fmax=7600)
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


class Wav2LipTestDataset(Dataset):
    """Full-length, deterministic test samples for external evaluation."""

    def __init__(self, metadata_path, split="test"):
        with open(metadata_path) as f:
            data = json.load(f)
        self.samples = [
            s for s in data
            if s["split"] == split and s["emotion_idx"] not in EXCLUDE
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav, _ = torchaudio.load(s["audio_path"])
        audio_1d = wav.squeeze(0)
        mel = wav_to_mel(s["audio_path"])

        frames = np.load(s["frames_path"]).astype(np.float32) / 255.0  # (T, H, W, 3)
        T = frames.shape[0]

        gt = torch.from_numpy(frames).permute(0, 3, 1, 2)
        if gt.shape[2] != IMG_SIZE or gt.shape[3] != IMG_SIZE:
            gt = F.interpolate(gt, size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                                align_corners=False)

        masked = gt.clone()
        masked[:, :, IMG_SIZE // 2:, :] = 0.0
        # Deterministic reference frame: use first frame
        ref = gt[0:1].expand(T, -1, -1, -1)
        face_input = torch.cat([ref, masked], dim=1)

        # Trim mel to exactly T frames
        need = MEL_STEP * T
        if mel.shape[1] < need:
            mel = np.pad(mel, ((0, 0), (0, need - mel.shape[1])))
        else:
            mel = mel[:, :need]
        mel_chunks = [
            torch.from_numpy(mel[:, t * MEL_STEP:(t + 1) * MEL_STEP]).unsqueeze(0)
            for t in range(T)
        ]
        mel_tensor = torch.stack(mel_chunks, dim=0)  # (T, 1, 80, 16)

        return {
            "sample_id": s["sample_id"],
            "mel": mel_tensor,
            "face_input": face_input,
            "gt": gt,
            "audio": audio_1d,
            "emotion": REMAP[s["emotion_idx"]],
            "n_frames": T,
        }


test_ds = Wav2LipTestDataset(METADATA, split="test")
print(f"Test samples: {len(test_ds)}")
from collections import Counter
print("Per-emotion counts:",
      dict(Counter(EMOTIONS[s["emotion"]] for s in test_ds)))

In [ ]:
def load_wav2lip(ckpt_path, device):
    model = Wav2LipModel()
    try:
        ckpt = torch.load(WAV2LIP_CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(WAV2LIP_CKPT, map_location="cpu")
    state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    # Overlay fine-tuned weights
    try:
        ft = torch.load(ckpt_path, map_location=device, weights_only=True)
    except TypeError:
        ft = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ft, strict=False)
    return model.to(device).eval()


@torch.no_grad()
def generate_full_video(model, sample):
    """Returns generated video tensor (T, 3, 96, 96) on CPU."""
    mel = sample["mel"].unsqueeze(0).to(DEVICE)  # (1, T, 1, 80, 16)
    face_in = sample["face_input"].unsqueeze(0).to(DEVICE)
    T = mel.shape[1]
    gens = []
    for t in range(T):
        g = model(mel[:, t], face_in[:, t])
        gens.append(g.squeeze(0).cpu())
    return torch.stack(gens, dim=0).clamp(0, 1)

## Per-sample external video prediction

Strategy: classify every 4th generated frame with the AffectNet ViT, average softmax probabilities, then take argmax **restricted to the 3 target classes** for per-sample prediction. Per-frame full-7-class logits are also kept for confusion analysis.

In [ ]:
FRAME_STRIDE = 4           # classify 1 of every N generated frames
EXT_BATCH = 32             # batch size for external classifier


@torch.no_grad()
def external_predict_video(gen_video):
    """gen_video: (T, 3, 96, 96) in [0, 1]. Returns:
       - mean_probs over sampled frames (7-dim AffectNet softmax)
       - predicted target-class label (0/1/2)
       - per-frame target-restricted argmax list
    """
    frames = gen_video[::FRAME_STRIDE]
    # Convert to PIL then let the processor handle resize/normalize
    pil_frames = [
        Image.fromarray((f.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
        for f in frames
    ]
    all_probs = []
    for i in range(0, len(pil_frames), EXT_BATCH):
        chunk = pil_frames[i:i + EXT_BATCH]
        inputs = ext_video_proc(chunk, return_tensors="pt").to(DEVICE)
        logits = ext_video_model(**inputs).logits
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu())
    probs = torch.cat(all_probs, dim=0)  # (F, 7)
    mean_probs = probs.mean(dim=0)

    # Per-frame restricted argmax (for voting analysis)
    tgt_indices = [TARGET_EXT_IDS[e] for e in EMOTIONS]
    restricted = probs[:, tgt_indices]  # (F, 3)
    per_frame_labels = restricted.argmax(dim=1).tolist()

    # Sample-level: argmax over 3 targets of mean probs
    sample_label = int(mean_probs[tgt_indices].argmax().item())
    return mean_probs.numpy(), sample_label, per_frame_labels

In [ ]:
def evaluate_checkpoint(name, ckpt_path):
    model = load_wav2lip(ckpt_path, DEVICE)
    rows = []
    for i in tqdm(range(len(test_ds)), desc=f"[{name}]", leave=False):
        sample = test_ds[i]
        gen = generate_full_video(model, sample)
        mean_probs, sample_label, per_frame = external_predict_video(gen)
        rows.append({
            "sample_id": sample["sample_id"],
            "emotion_true": sample["emotion"],
            "ext_pred": sample_label,
            "ext_probs_7": mean_probs.tolist(),
            "ext_per_frame": per_frame,
        })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    df.to_json(OUT_DIR / f"{name}.json", orient="records")
    return df


per_model = {}
for name, path in CHECKPOINTS:
    print(f"\n=== Evaluating {name} ===")
    per_model[name] = evaluate_checkpoint(name, path)
print("\nAll checkpoints evaluated.")

In [ ]:
def model_metrics(df):
    y = df["emotion_true"].to_numpy()
    p = df["ext_pred"].to_numpy()
    acc = accuracy_score(y, p)
    f1_macro = f1_score(y, p, labels=list(range(NUM_EMO)), average="macro", zero_division=0)
    per_f1 = f1_score(y, p, labels=list(range(NUM_EMO)), average=None, zero_division=0)
    return acc, f1_macro, per_f1

summary = []
for name in [n for n, _ in CHECKPOINTS]:
    acc, f1m, per = model_metrics(per_model[name])
    summary.append({
        "config": name,
        "ext_accuracy": acc,
        "ext_F1_macro": f1m,
        **{f"ext_F1_{EMOTIONS[i]}": per[i] for i in range(NUM_EMO)},
    })

summary_df = pd.DataFrame(summary)
base_f1 = summary_df.loc[summary_df["config"] == "baseline", "ext_F1_macro"].iloc[0]
summary_df["Δ_F1_vs_baseline"] = summary_df["ext_F1_macro"] - base_f1
summary_df = summary_df.sort_values("ext_F1_macro", ascending=False).reset_index(drop=True)

print("\n=== EXTERNAL (AffectNet ViT) evaluation on test split ===")
print(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)

In [ ]:
# McNemar vs baseline — did external emotion recognition meaningfully improve?
base_df = per_model["baseline"].set_index("sample_id")

rows = []
for name, _ in CHECKPOINTS:
    if name == "baseline":
        continue
    other = per_model[name].set_index("sample_id")
    shared = base_df.index.intersection(other.index)
    y = base_df.loc[shared, "emotion_true"].to_numpy()
    b_ok = (base_df.loc[shared, "ext_pred"].to_numpy() == y)
    e_ok = (other.loc[shared, "ext_pred"].to_numpy() == y)
    n01 = int((b_ok & ~e_ok).sum())  # baseline right, model wrong
    n10 = int((~b_ok & e_ok).sum())  # baseline wrong, model right
    chi2 = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
    p_val = 1 - stats.chi2.cdf(chi2, df=1) if (n01 + n10) > 0 else 1.0
    rows.append({
        "config": name,
        "correct_b→w": n01,
        "wrong_b→c":   n10,
        "McNemar χ²":  chi2,
        "McNemar p":   p_val,
        "significant (p<0.05)": p_val < 0.05,
    })

mcnemar_df = pd.DataFrame(rows)
print("\n=== McNemar (external emotion correctness vs baseline) ===")
print(mcnemar_df.to_string(index=False))
mcnemar_df.to_csv(OUT_DIR / "mcnemar.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot: external F1 per config, per-emotion breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
names = summary_df["config"].tolist()

axes[0].bar(names, summary_df["ext_F1_macro"], color="steelblue", alpha=0.85)
axes[0].axhline(base_f1, color="red", ls="--", label=f"baseline={base_f1:.3f}")
axes[0].set_title("External macro-F1 on generated test videos")
axes[0].set_ylabel("F1 (AffectNet ViT)")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend()

x = np.arange(NUM_EMO)
w = 0.8 / len(names)
for i, n in enumerate(names):
    vals = [summary_df.loc[summary_df["config"] == n, f"ext_F1_{e}"].iloc[0] for e in EMOTIONS]
    axes[1].bar(x + (i - len(names) / 2 + 0.5) * w, vals, w, label=n)
axes[1].set_xticks(x)
axes[1].set_xticklabels(EMOTIONS)
axes[1].set_ylabel("External F1")
axes[1].set_ylim(0, 1)
axes[1].set_title("External per-emotion F1")
axes[1].legend(fontsize=7, ncol=len(names))
plt.tight_layout()
plt.show()

# Confusion matrix for best config
best_name = summary_df.iloc[0]["config"]
best = per_model[best_name]
cm = confusion_matrix(best["emotion_true"], best["ext_pred"], labels=list(range(NUM_EMO)))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".2f",
             cmap="Blues", xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax)
ax.set_xlabel("External (AffectNet ViT) prediction")
ax.set_ylabel("Ground truth")
ax.set_title(f"External CM — {best_name} (test split)")
plt.tight_layout()
plt.show()

## Optional: dimensional audio check (MSP-Podcast wav2vec2)

Second external perspective — valence/arousal on the *original* audio. Used only to confirm that our test-split emotion labels align with what an independent audio model perceives. Not used to score generation (generation is video-only).

In [ ]:
RUN_AUDIO_SANITY = True

if RUN_AUDIO_SANITY:
    EXT_AUDIO_MODEL = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
    ext_audio_proc = AutoFeatureExtractor.from_pretrained(EXT_AUDIO_MODEL)
    ext_audio_model = AutoModel.from_pretrained(EXT_AUDIO_MODEL).to(DEVICE).eval()

    @torch.no_grad()
    def audio_vad(wav_1d):
        inputs = ext_audio_proc(wav_1d.numpy(), sampling_rate=SR, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        out = ext_audio_model(**inputs)
        # audeering model returns (arousal, dominance, valence) as last-hidden pooled
        vad = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
        return vad[:3]  # first 3 dims are (arousal, dominance, valence)

    rows = []
    for s in tqdm(test_ds, desc="External audio VAD"):
        vad = audio_vad(s["audio"])
        rows.append({"emotion": EMOTIONS[s["emotion"]],
                     "arousal": float(vad[0]),
                     "dominance": float(vad[1]),
                     "valence": float(vad[2])})
    vad_df = pd.DataFrame(rows)
    print("\n=== Ground-truth audio VAD (MSP-Podcast model) ===")
    print(vad_df.groupby("emotion")[["arousal", "dominance", "valence"]].mean().round(3))
    print("\nExpected (if labels agree with perception):")
    print("  happy   — high valence, mid arousal")
    print("  angry   — low valence, high arousal, high dominance")
    print("  disgust — low valence, mid-low arousal")

    del ext_audio_model
    torch.cuda.empty_cache()

## Thesis-ready combined table

Merges internal-encoder F1 (from ablation training runs) with external-classifier F1 from this notebook — the side-by-side answer to "does external evaluation confirm the gains?". Fill in the internal values from your `04b_wav2lip_loss_ablation.ipynb` run output.

In [ ]:
# Paste the internal val F1 from the 04b ablation output here:
INTERNAL_VAL_F1 = {
    "baseline":  None,   # fill from 04b result
    "cos-only":  None,
    "ce-only":   None,
    "ce-cos":    None,
    "ce-kl":     None,
}

combined = summary_df[["config", "ext_accuracy", "ext_F1_macro",
                         "ext_F1_happy", "ext_F1_angry", "ext_F1_disgust",
                         "Δ_F1_vs_baseline"]].copy()
combined["internal_val_F1"] = combined["config"].map(INTERNAL_VAL_F1)
combined = combined[["config", "internal_val_F1", "ext_F1_macro", "Δ_F1_vs_baseline",
                      "ext_accuracy", "ext_F1_happy", "ext_F1_angry", "ext_F1_disgust"]]
print("\n=== Internal vs External F1 (for thesis table) ===")
print(combined.to_string(index=False))
combined.to_csv(OUT_DIR / "internal_vs_external.csv", index=False)